In [1]:
import pandas as pd
import numpy as np

# 1. Load the sampled dataset
df = pd.read_csv("sampled_transactions.csv")
print(f"Total dataset shape: {df.shape}")

# 2. Select strictly non-leaking features for ML
feature_cols = ['step', 'type', 'amount']
target_col = 'isFraud'

X = df[feature_cols]
y = df[target_col]

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) distribution:\n{y.value_counts(normalize=True)}")

Total dataset shape: (5000, 11)
Features (X) shape: (5000, 3)
Target (y) distribution:
isFraud
0    0.8
1    0.2
Name: proportion, dtype: float64


In [2]:
from sklearn.model_selection import train_test_split

# First split: 70% Train, 30% Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: Split the 30% equally into 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training set:   {X_train.shape[0]} rows (Fraud: {y_train.sum()}, Normal: {len(y_train) - y_train.sum()})")
print(f"Validation set: {X_val.shape[0]} rows (Fraud: {y_val.sum()}, Normal: {len(y_val) - y_val.sum()})")
print(f"Test set:       {X_test.shape[0]} rows (Fraud: {y_test.sum()}, Normal: {len(y_test) - y_test.sum()})")

Training set:   3500 rows (Fraud: 700, Normal: 2800)
Validation set: 750 rows (Fraud: 150, Normal: 600)
Test set:       750 rows (Fraud: 150, Normal: 600)


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

numeric_features = ['step', 'amount']
categorical_features = ['type']

# Define preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Combine preprocessor and model into a single pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Train strictly on the training set
model_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully!")

Pipeline trained successfully!


In [ ]:
from sklearn.metrics import precision_recall_curve, auc, average_precision_score

# Predict probabilities on the validation set
# Column 1 is P(isFraud = 1)
val_probs = model_pipeline.predict_proba(X_val)[:, 1]

# Calculate PR-AUC using the validation set
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
val_pr_auc = auc(recalls, precisions)
print(f"Validation PR-AUC: {val_pr_auc:.4f}\n")

# precision_recall_curve returns one extra precision/recall point
# compared with the number of thresholds, so we use :-1 here.
f1_scores = (2 * precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1])
f1_scores = np.nan_to_num(f1_scores, nan=0.0)

best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]
    
print(f"Best Threshold: {best_threshold:.4f}\n")
print(f"Validation F1 at Best Threshold: {f1_scores[best_idx]:.4f}\n")

Validation PR-AUC: 0.7040

Best Threshold: 0.3570

Validation F1 at Best Threshold: 0.7192



In [5]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# Predict probabilities on the untouched test set
test_probs = model_pipeline.predict_proba(X_test)[:, 1]

# Test PR-AUC is reported without using the test set for tuning.
test_pr_auc = average_precision_score(y_test, test_probs)

# Apply the threshold chosen on the validation set
test_preds = (test_probs >= best_threshold).astype(int)

print(f"=== FINAL TEST EVALUATION (Threshold = {best_threshold:.3f}) ===\n")
print(f"Test PR-AUC: {test_pr_auc:.4f}\n")

print("Confusion Matrix:")
cm = confusion_matrix(y_test, test_preds)
print(f"TN: {cm[0,0]} | FP: {cm[0,1]}")
print(f"FN: {cm[1,0]} | TP: {cm[1,1]}\n")
    
print("Classification Report:")
print(classification_report(y_test, test_preds, target_names=["Normal (0)", "Fraud (1)" ]))


=== FINAL TEST EVALUATION (Threshold = 0.357) ===

Test PR-AUC: 0.7390

Confusion Matrix:
TN: 552 | FP: 48
FN: 46 | TP: 104

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.92      0.92      0.92       600
   Fraud (1)       0.68      0.69      0.69       150

    accuracy                           0.87       750
   macro avg       0.80      0.81      0.81       750
weighted avg       0.88      0.87      0.87       750



In [6]:
import joblib

model_artifact = {
    'pipeline': model_pipeline,
    'threshold': best_threshold,
    'features': feature_cols
}

joblib.dump(model_artifact, 'fraud_model.joblib')
print("Model artifact successfully saved to 'fraud_model.joblib'!")

Model artifact successfully saved to 'fraud_model.joblib'!
